# VGG16 Training on CIFAR-10 - Pure FP32 Baseline (No Acceleration)

Notebook ini melatih VGG16 pada CIFAR-10 menggunakan **Pure Float32 (FP32 / Single Precision)** standar tanpa akselerasi presisi rendah.

### Konsep Pure FP32:
- Semua parameter, bobot, aktivasi, dan input data diproses dalam format `torch.float32` (32-bit single precision).
- **TF32 dinonaktifkan secara eksplisit** (`allow_tf32 = False`) untuk mendapatkan nilai perbandingan baseline murni (unaccelerated FP32) yang stabil dan presisi tinggi secara matematis.
- Notebook ini berfungsi sebagai baseline utama (ground truth) untuk membandingkan performa waktu (latency), konsumsi memori (VRAM), dan akurasi (convergence) terhadap paradigma presisi lainnya (TF32, FP16, FP8, dan AMP).

In [ ]:
import torch
import time
import os

# Menonaktifkan TF32 secara eksplisit demi presisi baseline murni
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")
print(f"TF32 matmul diizinkan: {torch.backends.cuda.matmul.allow_tf32}")
print(f"TF32 cuDNN diizinkan: {torch.backends.cudnn.allow_tf32}")

In [ ]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Standarisasi Transform Data CIFAR-10
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = torchvision.datasets.CIFAR10(root='./dataset/cifar10', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./dataset/cifar10', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)

In [ ]:
import torch.nn as nn

# Arsitektur VGG16 Standar disesuaikan untuk CIFAR-10 (10 Kelas)
class VGG16_CIFAR10(nn.Module):
    def __init__(self):
        super(VGG16_CIFAR10, self).__init__()
        self.features = self._make_layers()
        self.classifier = nn.Sequential(
            nn.Linear(512, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 10)
        )

    def forward(self, x):
        out = self.features(x)
        out = out.view(out.size(0), -1)
        out = self.classifier(out)
        return out

    def _make_layers(self):
        cfg = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M']
        layers = []
        in_channels = 3
        for x in cfg:
            if x == 'M':
                layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
            else:
                layers += [
                    nn.Conv2d(in_channels, x, kernel_size=3, padding=1),
                    nn.BatchNorm2d(x),
                    nn.ReLU(True)
                ]
                in_channels = x
        return nn.Sequential(*layers)

In [ ]:
def evaluate(model, data_loader, criterion, device):
    model.eval()
    correct = 0
    total = 0
    running_loss = 0.0
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
    accuracy = 100.0 * correct / total
    avg_loss = running_loss / total
    return accuracy, avg_loss

In [ ]:
# Inisialisasi Model, Loss Function, dan Optimizer (SGD standar)
model = VGG16_CIFAR10().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
epochs = 5

In [ ]:
print("Memulai pelatihan dengan Pure FP32...")
for epoch in range(1, epochs + 1):
    # Reset statistik memori puncak GPU
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
        
    start_time = time.time()
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
    epoch_time = time.time() - start_time
    train_acc = 100.0 * correct / total
    train_loss = running_loss / total
    
    # Evaluasi test accuracy
    test_acc, test_loss = evaluate(model, test_loader, criterion, device)
    
    # Profiling GPU VRAM
    if device.type == 'cuda':
        peak_memory = torch.cuda.max_memory_allocated(device) / (1024 * 1024)  # Konversi ke Megabytes (MB)
        print(f"Epoch [{epoch}/{epochs}] - Time: {epoch_time:.2f}s - "
              f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% - "
              f"Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.2f}% - "
              f"Peak VRAM: {peak_memory:.2f} MB")
    else:
        print(f"Epoch [{epoch}/{epochs}] - Time: {epoch_time:.2f}s - "
              f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% - "
              f"Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.2f}% - "
              f"Peak VRAM: N/A (CPU)")